### Data exploration: PAWS-Wiki

## 1. Dataset Overview

(explicar porque hemos decidido usar este dataset)

In this course we decided to use **PAWS-Wiki (Paraphrase Adversaries from Word Scrambling)**, a dataset designed for binary paraphrase detection. Each instance consists of two sentences labelled according to whether they express the same meaning.

Unlike conventional paraphrase datasets, PAWS contains sentence pairs with high lexical overlap but potentially different meanings, making it particularly challenging for models that rely primarily on word occurrence and frequency.

This makes PAWS suitable for investigating how different text representations and classification architectures capture word order and semantic relationships.

### 1.1. Imports and Setup

The following libraries are used for dataset loading, exploratory data analysis, text tokenisation, and visualisation.

In [16]:
from datasets import load_dataset
import pandas as pd
import numpy as np

from nltk.tokenize import TreebankWordTokenizer
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

## 2. Dataset Loading

The PAWS-Wiki dataset is loaded from Hugging Face using the `labeled_final` configuration, which provides manually annotated sentence pairs.

The original training, validation, and test splits are preserved to ensure consistency with the benchmark and prevent information leakage.

### 2.1. Loading the Dataset

In [ ]:
dataset = load_dataset(
    "google-research-datasets/paws",
    "labeled_final"
)

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'sentence1', 'sentence2', 'label'],
        num_rows: 49401
    })
    test: Dataset({
        features: ['id', 'sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['id', 'sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
})

### 2.2. Dataset Structure and Split Sizes

We first examine the structure of the dataset, including the available splits, number of instances, and feature columns.

Each instance corresponds to a sentence pair rather than an individual sentence. The dataset provides three predefined splits, which will be preserved throughout the experiments.

In [ ]:
split_info = pd.DataFrame([
    {
        "Split": split_name,
        "Instances": len(split_data),
        "Columns": ", ".join(split_data.column_names)
    }
    for split_name, split_data in dataset.items()
])

display(split_info)

,Split,Instances,Columns
0,train,49401,"id, sentence1, sentence2, label"
1,test,8000,"id, sentence1, sentence2, label"
2,validation,8000,"id, sentence1, sentence2, label"


### 2.3. Sample Inspection

We inspect a few examples from each class to better understand the paraphrase detection task and the linguistic characteristics of the dataset.

In [15]:
train_df = dataset["train"].to_pandas()

display(train_df.head(5))

,id,sentence1,sentence2,label
0,1,"In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .","In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .",0
1,2,The NBA season of 1975 -- 76 was the 30th season of the National Basketball Association .,The 1975 -- 76 season of the National Basketball Association was the 30th season of the NBA .,1
2,3,"There are also specific discussions , public profile debates and project discussions .","There are also public discussions , profile specific discussions , and project discussions .",0
3,4,"When comparable rates of flow can be maintained , the results are high .",The results are high when comparable flow rates can be maintained .,1
4,5,It is the seat of Zerendi District in Akmola Region .,It is the seat of the district of Zerendi in Akmola region .,1


what means each label

## 3. Class Distribution

Class distribution is an important aspect of a classification dataset, as imbalanced classes may affect model training and evaluation.

We examine the number and proportion of paraphrase (`label = 1`) and non-paraphrase (`label = 0`) instances across the dataset splits. This analysis will help determine whether accuracy is an appropriate evaluation metric and establish a majority-class baseline.

In [17]:
# Count instances of each class in every split
class_counts = pd.DataFrame({
    split: pd.Series(data["label"]).value_counts()
    for split, data in dataset.items()
}).T

# Ensure both classes are present and ordered
class_counts = class_counts.reindex(columns=[0, 1], fill_value=0)

class_counts.columns = ["Non-paraphrase (0)", "Paraphrase (1)"]

# Calculate totals and class proportions
class_counts["Total"] = class_counts[
    ["Non-paraphrase (0)", "Paraphrase (1)"]
].sum(axis=1)

class_counts["Paraphrase (%)"] = (
    100 * class_counts["Paraphrase (1)"] / class_counts["Total"]
).round(2)

display(class_counts)

,Non-paraphrase (0),Paraphrase (1),Total,Paraphrase (%)
train,27572,21829,49401,44.19
test,4464,3536,8000,44.20
validation,4461,3539,8000,44.24


## 4. Sentence Length Analysis

Sentence length is a basic structural property that may influence classification performance. We examine the length of both sentences in each pair and the absolute difference between them.

The analysis aims to determine whether sentence length varies across classes and whether differences in length could provide a superficial signal for paraphrase detection.

### 4.1. Descriptive Statistics

We first compute sentence lengths and summarise their distributions separately for paraphrase and non-paraphrase pairs. Only the training split is used for this exploratory analysis.

In [22]:
# Compute sentence lengths using whitespace tokenisation
length_df = train_df[["label", "sentence1", "sentence2"]].copy()

length_df["len_s1"] = length_df["sentence1"].str.split().str.len()
length_df["len_s2"] = length_df["sentence2"].str.split().str.len()

# Average sentence length and absolute length difference
length_df["mean_length"] = (
    length_df["len_s1"] + length_df["len_s2"]
) / 2

length_df["length_diff"] = (
    length_df["len_s1"] - length_df["len_s2"]
).abs()

# Summary statistics by class
length_summary = length_df.groupby("label").agg(
    mean_length=("mean_length", "mean"),
    median_length=("mean_length", "median"),
    std_length=("mean_length", "std"),
    mean_abs_diff=("length_diff", "mean"),
    median_abs_diff=("length_diff", "median"),
    equal_length_pct=("length_diff", lambda x: 100 * (x == 0).mean())
).round(2)

length_summary.index = ["Non-paraphrase (0)", "Paraphrase (1)"]

display(length_summary)

,mean_length,median_length,std_length,mean_abs_diff,median_abs_diff,equal_length_pct
Non-paraphrase (0),21.35,21.5,5.42,0.99,1.0,36.17
Paraphrase (1),21.32,21.5,5.41,1.02,1.0,33.20


## 5. Lexical Overlap Analysis

PAWS-Wiki was designed to challenge paraphrase detection models by including sentence pairs with high lexical overlap but potentially different meanings. Understanding how lexical similarity varies across classes is therefore important for identifying the limitations of different text representations.

This section examines unigram and bigram overlap, exact Bag-of-Words equality, and their relationship with paraphrase labels. Beyond characterising the dataset, these analyses aim to motivate the selection of feature representations for the classification experiments.

### 5.1. Unigram and Bigram Jaccard Similarity

Jaccard similarity measures the overlap between two sets:

$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

Unigram Jaccard measures shared vocabulary regardless of word order, while bigram Jaccard considers adjacent token pairs and therefore captures some local ordering information.

For this exploratory analysis, sentences are lowercased and split on whitespace. This simple tokenisation strategy is used consistently across both measures, independently of the tokenizer selected for the classification models.

In [24]:
def tokenize(text):
    """Simple whitespace tokenisation for exploratory analysis."""
    return text.lower().split()


def get_ngrams(tokens, n=1):
    """Extract consecutive n-grams from a token sequence."""
    return list(zip(*(tokens[i:] for i in range(n))))


def jaccard_similarity(a, b):
    """Compute Jaccard similarity between two collections."""
    a, b = set(a), set(b)

    if not a and not b:
        return 1.0

    return len(a & b) / len(a | b)

In [25]:
# Prepare training pairs
lexical_df = train_df[
    ["sentence1", "sentence2", "label"]
].copy()

# Tokenise each sentence
tokens_s1 = lexical_df["sentence1"].apply(tokenize)
tokens_s2 = lexical_df["sentence2"].apply(tokenize)

# Compute unigram and bigram Jaccard for each pair
lexical_df["unigram_jaccard"] = [
    jaccard_similarity(get_ngrams(t1, 1), get_ngrams(t2, 1))
    for t1, t2 in zip(tokens_s1, tokens_s2)
]

lexical_df["bigram_jaccard"] = [
    jaccard_similarity(get_ngrams(t1, 2), get_ngrams(t2, 2))
    for t1, t2 in zip(tokens_s1, tokens_s2)
]

# Summarise by class
jaccard_summary = lexical_df.groupby("label").agg(
    unigram_mean=("unigram_jaccard", "mean"),
    unigram_median=("unigram_jaccard", "median"),
    unigram_std=("unigram_jaccard", "std"),
    bigram_mean=("bigram_jaccard", "mean"),
    bigram_median=("bigram_jaccard", "median"),
    bigram_std=("bigram_jaccard", "std")
).round(3)

jaccard_summary = jaccard_summary.rename(index={
    0: "Non-paraphrase (0)",
    1: "Paraphrase (1)"
})

display(jaccard_summary)

,unigram_mean,unigram_median,unigram_std,bigram_mean,bigram_median,bigram_std
label,,,,,,
Non-paraphrase (0),0.884,0.882,0.074,0.499,0.500,0.151
Paraphrase (1),0.886,0.882,0.063,0.606,0.607,0.155


### 5.2. Exact Bag-of-Words Equality

Unigram Jaccard similarity ignores token frequencies and word order. To investigate the limitations of unigram-based representations more directly, we examine the proportion of sentence pairs with identical Bag-of-Words representations.

Two sentences have identical Bag-of-Words representations when they contain the same tokens with the same frequencies, regardless of their order.

We also identify pairs with identical token frequencies but different token sequences, where word-order information is necessarily lost.

In [26]:
# Identify pairs with identical token counts
lexical_df["same_bow"] = [
    Counter(t1) == Counter(t2)
    for t1, t2 in zip(tokens_s1, tokens_s2)
]

# Identify pairs with the same BoW but different token order
lexical_df["same_bow_diff_order"] = [
    Counter(t1) == Counter(t2) and t1 != t2
    for t1, t2 in zip(tokens_s1, tokens_s2)
]

# Compute proportions by class
bow_summary = lexical_df.groupby("label").agg(
    total_pairs=("label", "size"),
    same_bow_pct=("same_bow", lambda x: 100 * x.mean()),
    reordered_pairs_pct=(
        "same_bow_diff_order",
        lambda x: 100 * x.mean()
    )
).round(2)

bow_summary = bow_summary.rename(index={
    0: "Non-paraphrase (0)",
    1: "Paraphrase (1)"
})

display(bow_summary)

,total_pairs,same_bow_pct,reordered_pairs_pct
label,,,
Non-paraphrase (0),27572,8.52,8.46
Paraphrase (1),21829,3.94,3.75
